# Week 3 - Lab 11: Guided Chains and Conversational Memory

**Track:** AI Engineering Academy (Lowe's cohort) - Prompt chaining and memory with LangChain
**Format:** hands-on lab - build two production-shaped pipelines and reason about the trade-offs
**Runs fully offline.** A deterministic scripted model stands in for the LLM so every result is
reproducible and every check is stable. The optional appendix shows how to point the same code
at a real endpoint.

## What you will build

- **Part B - a three-step chain:** extract -> validate and repair -> summarize. You wire the
  stages with LCEL, gate the output with a JSON Schema, and let a repair step fix the one lead
  that comes back malformed.
- **Part C - a conversational assistant with memory:** you implement buffer memory and summary
  memory, a reset control, and lightweight telemetry, then compare recall against token cost.

## Outcomes

By the end you can:
1. Compose a multi-stage LCEL pipeline where each stage is a plain function.
2. Validate model output against a JSON Schema and drive a conditional repair loop.
3. Implement buffer and summary memory by hand and explain the recall-versus-cost trade-off.
4. Add a reset control and capture latency and approximate token cost.

## Time budget (about 165 minutes)

- Part A setup: 10 min
- Part B chain: 70 min
- Part C memory: 60 min
- Stretch: 25 min

Each part ends in soft checks that print PASS, FAIL, or ERROR. A clean Restart-and-Run-All gives
the true totals. There are no inline hints; if you get stuck, open HINTS.md for a graded nudge.

## Part A - Setup

Two provided cells give you everything the lab is built around: a deterministic model, the
synthetic Cordwell Pro data, the Lead schema, and the self-check harness. Read them, then leave
them alone. Your work starts in Part B.

> **On the model.** `ScriptedChatModel` is a real `langchain_core` chat model whose replies are
> fixed functions of the prompt. That is deliberate: it makes the pipeline mechanics - not a
> model's mood - the thing under test. The wiring you write is identical to what you would use
> against a live model.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# PROVIDED. Standard imports and a fixed, offline setup. Do not edit.
import json, re, time
from importlib.metadata import version
for pkg in ("langchain-core", "jsonschema", "rapidfuzz"):
    print(pkg, version(pkg))
from jsonschema import Draft202012Validator
from langchain_core.runnables import RunnableLambda

In [ ]:
"""LABKIT: provided infrastructure for Week03_Lab11 (Cordwell Pro trade desk).
Deterministic, offline. Students do NOT edit this; they build the chain/memory around it.
"""
from __future__ import annotations
import json, re, time
from typing import Any
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult

# ---- Synthetic data (clearly fictional). Cordwell Pro = Cordwell's trade/contractor desk. ----
LEADS = [
  {"id":"L01","email":"Subject: Pro pricing for 50 accounts. Body: Hi, this is Dana at BlueOrbit Facilities. We want pricing for ~50 pro accounts. We use Okta SSO and require SOC 2. Phone: +1-415-555-0199."},
  {"id":"L02","email":"Subject: EU rollout. Body: Bonjour, Novalie Property Group, based in Paris. We need EU data residency and Google Workspace SSO. Timeline: Q4."},
  {"id":"L03","email":"Subject: Trial to paid. Body: Marco from Apex Retail Group. Trial ends 10/31. We want 20 buyer seats and 80 viewer seats, SAML required, annual billing."},
  {"id":"L04","email":"Subject: Feature question. Body: Hello, does Cordwell Pro support bulk order export? Also interested in the nonprofit discount. Org: RiverAid."},
]

# The scripted model returns these canned extractions keyed by lead id.
# L02 is deliberately incomplete (missing company + need) so the repair step must fire.
_CANNED = {
  "L01": {"id":"L01","company":"BlueOrbit Facilities","need":"pricing","security":["SSO: Okta","SOC 2"],"seat_counts":{"editor":50,"viewer":0}},
  "L02": {"id":"L02","requirements":["EU data residency","Google Workspace SSO"],"timeline":"Q4"},
  "L03": {"id":"L03","company":"Apex Retail Group","need":"trial_to_paid","security":["SAML"],"seat_counts":{"editor":20,"viewer":80},"timeline":"trial ends 10/31","notes":"annual billing"},
  "L04": {"id":"L04","company":"RiverAid","need":"feature_question","requirements":["bulk order export","nonprofit discount"]},
}
# What the repair model returns for L02 once told what is missing.
_REPAIRED = {
  "L02": {"id":"L02","company":"Novalie Property Group","need":"eu_residency","requirements":["EU data residency","Google Workspace SSO"],"timeline":"Q4"},
}

ONBOARDING = {
  "turns": [
    {"role":"user","text":"Hi, I am Riley from Northshore Facilities, onboarding to Cordwell Pro."},
    {"role":"assistant","text":"Welcome Riley. How can I help set up your Cordwell Pro account?"},
    {"role":"user","text":"We need SSO via Azure AD and MFA for all admins."},
    {"role":"assistant","text":"Noted. SSO via Azure AD and MFA for admins."},
    {"role":"user","text":"Please provision four sites: Northside Depot, Westgate Depot, Fleet, and Back Office."},
    {"role":"assistant","text":"Understood. Four sites: Northside Depot, Westgate Depot, Fleet, Back Office."},
    {"role":"user","text":"Our billing contact is ap@northshore.example and terms are net 30."},
    {"role":"assistant","text":"Recorded billing contact and net 30 terms."},
  ],
  "final_q": "Remind me: what SSO did we choose and which four sites did I ask for?",
  "eval": {"expected_sso":"Azure AD","expected_sites":["Northside Depot","Westgate Depot","Fleet","Back Office"]},
}

LEAD_SCHEMA = {
  "type":"object",
  "properties":{
    "id":{"type":"string"},
    "company":{"type":"string"},
    "need":{"type":"string","enum":["pricing","eu_residency","trial_to_paid","feature_question","other"]},
    "requirements":{"type":"array","items":{"type":"string"}},
    "security":{"type":"array","items":{"type":"string"}},
    "seat_counts":{"type":"object","properties":{"editor":{"type":"integer"},"viewer":{"type":"integer"}}},
    "timeline":{"type":"string"},
    "notes":{"type":"string"},
  },
  "required":["id","company","need"],
  "additionalProperties": False,
}

class ScriptedChatModel(BaseChatModel):
    """Deterministic, offline stand-in for a chat LLM.

    Routes on contract markers in the prompt to one of four roles: extract,
    repair, summarize, or converse. No network, fully reproducible.
    """
    @property
    def _llm_type(self) -> str: return "scripted-chat-model"
    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        text = "\n".join(m.content for m in messages)
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=self._route(text)))])
    def _route(self, text: str) -> str:
        if "JSON repair assistant" in text: return self._repair(text)
        if "Sales Ops brief" in text:       return self._summarize(text)
        if "information extraction agent" in text: return self._extract(text)
        if "Condense the conversation" in text: return self._condense(text)
        return self._chat(text)
    def _lead_id(self, text: str) -> str:
        m = re.search(r"\b(L0\d)\b", text); return m.group(1) if m else "L00"
    def _extract(self, text: str) -> str:
        return "```json\n" + json.dumps(_CANNED.get(self._lead_id(text), {"id": self._lead_id(text)})) + "\n```"
    def _repair(self, text: str) -> str:
        lid = self._lead_id(text)
        return json.dumps(_REPAIRED.get(lid, {**_CANNED.get(lid, {"id":lid}), "company":"Unknown","need":"other"}))
    def _summarize(self, text: str) -> str:
        try: lead = json.loads(re.search(r"\{.*\}", text, re.S).group(0))
        except Exception: lead = {}
        lines = [f"**Company and Need** {lead.get('company','Unknown')} needs {lead.get('need','other')}."]
        sec = lead.get("security") or []; req = lead.get("requirements") or []
        if sec or req: lines.append("**Key items** " + ", ".join(sec + req))
        sc = lead.get("seat_counts") or {}
        if sc: lines.append(f"**Seats** editor={sc.get('editor',0)}, viewer={sc.get('viewer',0)}")
        if lead.get("timeline"): lines.append(f"**Timeline** {lead['timeline']}")
        return "\n".join(lines)
    def _chat(self, text: str) -> str:
        # Only recall when explicitly asked; otherwise just acknowledge the current turn.
        is_recall = any(k in text for k in ["Remind me", "what SSO", "which", "how many", "?"])
        if not is_recall:
            return "Got it, recorded."
        sso = "Azure AD" if "Azure AD" in text else ""
        sites = [s for s in ["Northside Depot","Westgate Depot","Fleet","Back Office"] if s in text]
        bits = []
        if sso: bits.append(f"SSO is {sso}")
        if sites: bits.append("sites are " + ", ".join(sites))
        return ("From our notes, " + "; ".join(bits) + ".") if bits else "I do not have that in memory yet."
    def _condense(self, text: str) -> str:
        facts = []
        if "Azure AD" in text: facts.append("SSO=Azure AD" + (" with MFA for admins" if "MFA" in text else ""))
        sites = [s for s in ["Northside Depot","Westgate Depot","Fleet","Back Office"] if s in text]
        if sites: facts.append("sites=" + ", ".join(sites))
        if "net 30" in text: facts.append("billing=net 30")
        return "Key facts: " + "; ".join(facts) + "." if facts else "No durable facts yet."

def approx_tokens(text: str) -> int:
    return 0 if not text else max(1, round(len(text)/4))

In [ ]:
# PROVIDED. Prompt templates for the three-step chain. Do not edit.
from langchain_core.prompts import ChatPromptTemplate

EXTRACT_TPL = ChatPromptTemplate.from_template(
    "You are a precise information extraction agent for the Cordwell Pro trade desk.\n"
    "Return STRICT JSON matching the schema. Include only facts that are present; do not invent.\n"
    "<SCHEMA>{schema}</SCHEMA>\n<EMAIL>{id}: {email}</EMAIL>\n"
    "<OUTPUT_CONTRACT>Return only the JSON object.</OUTPUT_CONTRACT>")

REPAIR_TPL = ChatPromptTemplate.from_template(
    "You are a JSON repair assistant.\nGiven a possibly invalid Lead JSON plus the validation errors, "
    "return a corrected object that satisfies the schema. Keep correct values; fill only what is missing.\n"
    "<SCHEMA>{schema}</SCHEMA>\n<ERRORS>{errors}</ERRORS>\n<ORIGINAL>{original}</ORIGINAL>\nid={id}\n"
    "<OUTPUT_CONTRACT>Return only the repaired JSON object.</OUTPUT_CONTRACT>")

SUMMARY_TPL = ChatPromptTemplate.from_template(
    "You are an executive assistant writing a Cordwell Pro Sales Ops brief.\n"
    "From the validated Lead JSON, write a compact Markdown brief (company and need, key items, "
    "seats, timeline). Keep it short.\n<LEAD>{lead}</LEAD>\n"
    "<OUTPUT_CONTRACT>Return only Markdown.</OUTPUT_CONTRACT>")

MODEL = ScriptedChatModel()
print("Templates ready. Model:", MODEL._llm_type)

In [ ]:
# PROVIDED. Soft self-check harness. It never hard-crashes: an unimplemented
# stub is reported as ERROR, a wrong answer as FAIL, a correct answer as PASS.
# Do not edit.
_RESULTS = []

def check(name, fn):
    """Run a check function that returns (ok: bool, detail: str)."""
    try:
        ok, detail = fn()
        status = "PASS" if ok else "FAIL"
    except NotImplementedError:
        status, detail = "ERROR", "not implemented yet"
    except Exception as exc:
        status, detail = "ERROR", f"{type(exc).__name__}: {exc}"
    _RESULTS.append((name, status))
    print(f"[{status}] {name}  ::  {detail}")

def tally():
    from collections import Counter
    c = Counter(s for _, s in _RESULTS)
    print(f"\nTOTALS  pass={c['PASS']}  fail={c['FAIL']}  error={c['ERROR']}  (of {len(_RESULTS)})")
    return c

print("check() harness ready.")

## Part B - Three-step chain: extract, validate and repair, summarize

**Goal.** Turn four raw lead emails into validated Lead records and short Sales Ops briefs.

The shape is `extract -> repair -> summarize`, composed with LCEL. The extract stage returns
JSON, the repair stage runs only when the JSON fails the schema, and the summarize stage writes
the brief. One of the four leads comes back malformed on purpose so you can see the repair loop
do real work.

### TODO 1 - `as_json`

Model replies are text. You need a parser that survives code fences and never raises. Implement
to the contract in the docstring.

In [ ]:
def as_json(text: str) -> dict:
    """Parse a model reply into a dict.

    Contract:
      - Accept an optionally fenced JSON payload (it may be wrapped in a
        ```json ... ``` block or returned bare).
      - Return the parsed dict on success.
      - Return an empty dict on any parse failure (never raise).
    """
    raise NotImplementedError("Implement as_json.")

### TODO 2 - `schema_errors`

A dict that parsed cleanly is not necessarily valid. Return every schema violation as a readable
string, or an empty list when the object is valid. This is the gate that decides whether the
repair stage runs.

In [ ]:
def schema_errors(obj: dict) -> list[str]:
    """Return a list of human-readable schema violations for obj against LEAD_SCHEMA.

    Contract:
      - Use a Draft 2020-12 validator and collect every violation (not just the first).
      - Each item is a readable string that names the offending path and the message.
      - Return an empty list when obj fully satisfies LEAD_SCHEMA.
    """
    raise NotImplementedError("Implement schema_errors.")

### TODO 3 to 5 - the three stages

Implement `extract_step`, `repair_step`, and `summarize_step`. The repair stage is conditional:
it must be a no-op when the extracted JSON already validates, and must otherwise send the errors
back to the model for a fix.

In [ ]:
def extract_step(rec: dict) -> dict:
    """Stage 1. Prompt the model to extract a Lead from one email record.

    Contract:
      - Format EXTRACT_TPL with the serialized schema, the record id and email.
      - Invoke MODEL and parse the reply with as_json.
      - Return {"rec": rec, "json": <parsed dict>}.
    """
    raise NotImplementedError("Implement extract_step.")

def repair_step(state: dict) -> dict:
    """Stage 2. Repair only when the extracted JSON fails the schema.

    Contract:
      - Compute schema_errors on state["json"].
      - If there are none, return state unchanged plus {"errors": [], "repaired": False}.
      - Otherwise prompt the model with REPAIR_TPL (schema, errors, original, id),
        re-parse, and return the repaired json plus {"errors": <errs>, "repaired": True}.
    """
    raise NotImplementedError("Implement repair_step.")

def summarize_step(state: dict) -> dict:
    """Stage 3. Turn the validated Lead into a Markdown brief.

    Contract:
      - Format SUMMARY_TPL with the serialized validated json, invoke MODEL.
      - Return state plus {"md": <stripped markdown string>}.
    """
    raise NotImplementedError("Implement summarize_step.")

### TODO 6 - compose the chain

Wire the three stages into a single LCEL pipeline with `RunnableLambda` and the `|` operator,
then run it across all four leads.

In [ ]:
def build_lead_chain():
    """Compose the three stages into one LCEL pipeline.

    Contract:
      - Return extract_step | repair_step | summarize_step, wired with RunnableLambda,
        so the composed object exposes .invoke(record) -> final state dict.
    """
    raise NotImplementedError("Implement build_lead_chain.")

try:
    lead_chain = build_lead_chain()
    results = [lead_chain.invoke(rec) for rec in LEADS]
    for st in results:
        print(f"{st['rec']['id']}  repaired={st['repaired']!s:5s}  need={st['json'].get('need')}")
except Exception as exc:
    results = None
    print("Chain not ready:", exc)

In [ ]:
# Part B self-checks. Run after implementing the functions above.
def _c_as_json():
    a = as_json('```json\n{"id":"L01","company":"X","need":"pricing"}\n```')
    b = as_json("not json at all")
    return (a.get("company") == "X" and b == {}, f"fenced->{bool(a)} garbage->{b}")

def _c_schema_errors():
    good = {"id": "L1", "company": "Acme", "need": "pricing"}
    bad = {"id": "L1"}
    return (schema_errors(good) == [] and len(schema_errors(bad)) >= 1,
            f"good_errs={len(schema_errors(good))} bad_errs={len(schema_errors(bad))}")

def _c_extract():
    st = extract_step({"id": "L01", "email": LEADS[0]["email"]})
    return (st["json"].get("company") == "BlueOrbit Facilities", f"company={st['json'].get('company')!r}")

def _c_repair_fires():
    st = repair_step(extract_step({"id": "L02", "email": LEADS[1]["email"]}))
    return (st["repaired"] is True and schema_errors(st["json"]) == [],
            f"repaired={st['repaired']} need={st['json'].get('need')!r}")

def _c_all_valid():
    assert results is not None
    bad = [r["rec"]["id"] for r in results if schema_errors(r["json"])]
    return (bad == [], f"invalid={bad}")

def _c_briefs():
    assert results is not None
    ok = all(isinstance(r.get("md"), str) and len(r["md"]) > 0 for r in results)
    return (ok and len(results) == 4, f"briefs={len(results)}")

check("as_json handles fenced and garbage", _c_as_json)
check("schema_errors flags missing required", _c_schema_errors)
check("extract_step pulls company for L01", _c_extract)
check("repair_step fires and fixes L02", _c_repair_fires)
check("all four leads validate", _c_all_valid)
check("four Markdown briefs produced", _c_briefs)

**Checkpoint.** L02 should report `repaired=True` and still end valid; the other three should
pass on the first try. If `repair_step` fires on more than one lead, your `schema_errors` gate is
probably too strict.

## Part C - Conversational memory: buffer versus summary

**Goal.** Drive a scripted onboarding conversation, then ask the assistant to recall two facts it
was told earlier: the SSO choice and the four sites. You implement two memory strategies and one
control, then compare what each remembers against what each costs.

- **Buffer memory** keeps every turn. Perfect recall, cost grows with the conversation.
- **Summary memory** keeps a short verbatim window and compresses older turns into a running
  summary. Cheaper, at the risk of losing detail.
- **Reset** clears memory mid-session. A real privacy and topic-hygiene control.

In [ ]:
# PROVIDED. Chat and condense templates for the memory section. Do not edit.
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

CHAT_TPL = ChatPromptTemplate.from_messages([
    ("system", "You are the Cordwell Pro onboarding assistant. Cite remembered facts explicitly."),
    ("placeholder", "{history}"),
    ("human", "{input}"),
])

CONDENSE_TPL = ChatPromptTemplate.from_template(
    "Condense the conversation into durable key facts, one short line.\n{blob}")

print("Chat templates ready. Script turns:", len(ONBOARDING["turns"]))

### TODO 7 - `BufferMemory`

The simplest strategy. Keep every message; hand the whole list back on `load()`.

In [ ]:
class BufferMemory:
    """Keep every turn verbatim.

    Contract:
      - load() returns the full message list (list of BaseMessage), oldest first.
      - save(user, ai) appends a HumanMessage then an AIMessage.
      - reset() empties the memory.
    """
    def __init__(self):
        raise NotImplementedError("Implement BufferMemory.")
    def load(self):
        raise NotImplementedError("Implement BufferMemory.load.")
    def save(self, user, ai):
        raise NotImplementedError("Implement BufferMemory.save.")
    def reset(self):
        raise NotImplementedError("Implement BufferMemory.reset.")

### TODO 8 - `SummaryMemory`

Keep the last `keep_last` messages verbatim and fold everything older into a running summary by
calling the model with `CONDENSE_TPL`. This is where the recall-versus-cost trade-off lives.

In [ ]:
class SummaryMemory:
    """Keep the most recent turns verbatim and compress older ones into a running summary.

    Contract:
      - __init__(model, keep_last=2): store the model and the verbatim window size.
      - load() returns [SystemMessage(summary)] (only when a summary exists) followed by
        the last keep_last messages.
      - save(user, ai) appends the exchange; if the verbatim buffer now exceeds keep_last,
        evict the overflow, ask the model (CONDENSE_TPL) to fold the previous summary plus
        the evicted text into a new summary, and retain only the last keep_last messages.
      - reset() clears both the buffer and the summary.
    """
    def __init__(self, model, keep_last=2):
        raise NotImplementedError("Implement SummaryMemory.")
    def load(self):
        raise NotImplementedError("Implement SummaryMemory.load.")
    def save(self, user, ai):
        raise NotImplementedError("Implement SummaryMemory.save.")
    def reset(self):
        raise NotImplementedError("Implement SummaryMemory.reset.")

### TODO 9 - `run_session`

Thread a memory object through the scripted conversation, then ask the recall question. Capture
the final reply, the number of loaded messages, an approximate token count, and the latency of
the final call.

In [ ]:
def run_session(memory, reset_before_final=False):
    """Replay the scripted user turns through the model, then ask the recall question.

    Contract:
      - Start from a clean memory (call reset()).
      - For each scripted user turn, invoke MODEL with CHAT_TPL(history=memory.load(),
        input=<user text>) and save (user, reply) to memory.
      - If reset_before_final, clear memory just before the final question.
      - Ask ONBOARDING["final_q"], time only that final call (time.perf_counter).
      - Return {"final", "loaded_msgs", "approx_tokens", "latency"} where approx_tokens
        sums approx_tokens over the loaded history text, the final question, and the reply.
    """
    raise NotImplementedError("Implement run_session.")

try:
    buffer_run = run_session(BufferMemory())
    summary_run = run_session(SummaryMemory(MODEL, keep_last=2))
    print("buffer :", buffer_run["loaded_msgs"], "msgs,", buffer_run["approx_tokens"], "approx tokens")
    print("summary:", summary_run["loaded_msgs"], "msgs,", summary_run["approx_tokens"], "approx tokens")
except Exception as exc:
    buffer_run = summary_run = None
    print("Sessions not ready:", exc)

### TODO 10 - the reset control

`run_session` already accepts `reset_before_final`. Run a buffer session with the reset engaged
and watch recall disappear - the assistant answers the final question with an empty history.

In [ ]:
try:
    reset_run = run_session(BufferMemory(), reset_before_final=True)
    print("reset reply:", reset_run["final"])
except Exception as exc:
    reset_run = None
    print("Reset run not ready:", exc)

In [ ]:
# Part C self-checks.
from rapidfuzz import fuzz
EXP_SSO = ONBOARDING["eval"]["expected_sso"]
EXP_SITES = ONBOARDING["eval"]["expected_sites"]

def recalls(text, target, thr=80):
    return fuzz.partial_ratio(target.lower(), (text or "").lower()) >= thr

def _c_buffer_recall():
    assert buffer_run is not None
    ok = recalls(buffer_run["final"], EXP_SSO) and all(recalls(buffer_run["final"], s) for s in EXP_SITES)
    return (ok, f"reply={buffer_run['final'][:60]!r}")

def _c_summary_recall():
    assert summary_run is not None
    ok = recalls(summary_run["final"], EXP_SSO) and all(recalls(summary_run["final"], s) for s in EXP_SITES)
    return (ok, f"msgs={summary_run['loaded_msgs']} reply={summary_run['final'][:50]!r}")

def _c_summary_compresses():
    assert buffer_run is not None and summary_run is not None
    return (summary_run["approx_tokens"] < buffer_run["approx_tokens"],
            f"summary={summary_run['approx_tokens']} < buffer={buffer_run['approx_tokens']}")

def _c_reset_loses():
    assert reset_run is not None
    lost = not recalls(reset_run["final"], EXP_SSO)
    return (lost and reset_run["loaded_msgs"] == 0, f"reply={reset_run['final'][:50]!r}")

def _c_telemetry():
    assert buffer_run is not None
    return (isinstance(buffer_run["latency"], float) and isinstance(buffer_run["approx_tokens"], int)
            and buffer_run["approx_tokens"] > 0, f"latency={type(buffer_run['latency']).__name__}")

check("buffer recalls SSO and all sites", _c_buffer_recall)
check("summary still recalls after compression", _c_summary_recall)
check("summary uses fewer tokens than buffer", _c_summary_compresses)
check("reset before final wipes recall", _c_reset_loses)
check("telemetry captured (latency + tokens)", _c_telemetry)

**The trade-off, in three runs.**

| Strategy | Loaded messages | Approx tokens | Recalls SSO and sites |
|---|---|---|---|
| Buffer | 8 | 127 | yes |
| Summary (keep_last=2) | 3 | 88 | yes |
| Buffer + reset | 0 | 25 | no |

Summary buys back about a third of the tokens here and still answers correctly, because the
condense step preserves the keywords. That will not always hold: a lossy summary can drop a
detail the user later asks for. Buffer never forgets but pays linearly. Reset is the honest way
to make the assistant forget on purpose.

## Stretch (optional)

Two independent extensions. Solutions are in the instructor key.

### Stretch 1 - deterministic pre-normalizer

Do not ask a model to parse what a regex can nail. Implement `prenormalize` to pull the phone
number, seat counts, and known security tokens straight from the raw email. In production you
would feed these as a trusted context field to the extract prompt so the model never has to guess
a number.

In [ ]:
def prenormalize(email: str) -> dict:
    """Deterministic, model-free pre-pass: pull facts a regex can nail reliably.

    Contract:
      - phone: first phone-like run of digits (with optional + and separators).
      - seat_hint: {role: count} for patterns like "20 buyer seats", "80 viewer".
      - security: append "SOC 2", "SAML", "SSO: Okta" when those tokens appear.
      - Omit keys that are not found. Never raise.
    """
    raise NotImplementedError("Implement prenormalize.")

### Stretch 2 - window memory

Add a third strategy that keeps only the last k exchanges. It is the cheapest of all, and it
exposes the failure mode of naive truncation: implement it, run it, and watch which fact it
silently drops.

In [ ]:
class WindowMemory:
    """Keep only the last k exchanges (2*k messages). Cheap, but silently forgets.

    Contract:
      - __init__(k=2): window of k exchanges.
      - load() returns the last 2*k messages.
      - save(user, ai) appends the exchange (no trimming needed on save).
      - reset() empties the memory.
    """
    def __init__(self, k=2):
        raise NotImplementedError("Implement WindowMemory.")
    def load(self):
        raise NotImplementedError("Implement WindowMemory.load.")
    def save(self, user, ai):
        raise NotImplementedError("Implement WindowMemory.save.")
    def reset(self):
        raise NotImplementedError("Implement WindowMemory.reset.")

try:
    window_run = run_session(WindowMemory(k=2))
    print("window reply:", window_run["final"])
except Exception as exc:
    window_run = None
    print("Window run not ready:", exc)

In [ ]:
# Stretch self-checks.
def _c_prenorm():
    a = prenormalize(LEADS[0]["email"])
    b = prenormalize(LEADS[2]["email"])
    ok = a.get("phone", "").endswith("0199") and b.get("seat_hint", {}).get("viewer") == 80
    return (ok, f"L01_phone={a.get('phone')!r} L03_seats={b.get('seat_hint')}")

def _c_window_partial():
    assert window_run is not None
    keeps_sites = all(recalls(window_run["final"], s) for s in EXP_SITES)
    drops_sso = not recalls(window_run["final"], EXP_SSO)
    return (keeps_sites and drops_sso, f"sites_kept={keeps_sites} sso_dropped={drops_sso}")

check("prenormalize extracts phone and seats", _c_prenorm)
check("window memory silently drops the SSO fact", _c_window_partial)
tally()

## Appendix - pointing at a real model (inert, not run in the graded path)

The graded lab is offline by design. To see the same pipeline against a live model, swap the one
line that builds `MODEL`. Any OpenAI-compatible endpoint works: a local server (LM Studio or
Ollama) or a hosted provider. Keep the key in an environment variable; never hard-code it. Note
that a real model makes outputs non-deterministic, so the exact-recall checks become approximate.

```python
# import os
# from langchain_openai import ChatOpenAI
# MODEL = ChatOpenAI(
#     model=os.environ.get("LAB_MODEL", "local-model"),
#     base_url=os.environ.get("OPENAI_BASE_URL", "http://localhost:1234/v1"),
#     api_key=os.environ.get("OPENAI_API_KEY", "not-needed-for-local"),
#     temperature=0.2, timeout=45, max_retries=0,
# )
```

**Responsible AI note.** All data here is synthetic and clearly fictional. When you move to a
real model, the same validate-and-repair gate in Part B is your first line of defense against
malformed or injected output, and the reset control in Part C is a real privacy lever.